In [20]:
import matplotlib.pyplot as plt
import numpy as np
import math
import random
import torch
from PIL import Image
import open_clip
import os

device = "cuda"

def cosDist(data1, data2):
  cos = torch.nn.CosineSimilarity(dim=0, eps=1e-6)
  return 1 - cos(data1, data2)

def cosDistTensor(data1, data2):
  cos = torch.nn.CosineSimilarity(dim=0, eps=1e-6)
  return 1 - cos(data1, data2)

def getImagePathsAndFilenames(dir):
  imagePaths = []
  filenames = os.listdir(dir)
  for filename in filenames:
      imagePaths.append(os.path.join(dir, filename))
  return imagePaths, filenames

def getImageFeatures(imagePaths):
  model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k', device=device)

  imagePaths, filenames = getImagePathsAndFilenames(dir)

  images = [preprocess(Image.open(path)).unsqueeze(0).to(device) for path in imagePaths]

  with torch.no_grad(), torch.cuda.amp.autocast():
      imageFeatures = [model.encode_image(image) for image in images]
      return torch.cat(imageFeatures, 0)
  
def getTextFeatures(texts):
  model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k', device=device)
  tokenizer = open_clip.get_tokenizer('ViT-B-32')
  tokens = tokenizer(texts).to(device)

  with torch.no_grad(), torch.cuda.amp.autocast():
    text_features = model.encode_text(tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    return text_features

In [21]:
dir = 'D:\\school\\videa\\images'
imagePaths, filenames = getImagePathsAndFilenames(dir)
#classes = ["collie", "dachshund", "street dog", "bolognese dog", "bichon frisé", "maltese dog", "woman lying on a bed with her dog"]
classes = ["collie"]
imageFeatures = getImageFeatures(imagePaths)
textFeatures = getTextFeatures(classes)

text_probs = (100.0 * imageFeatures @ textFeatures.T).softmax(dim=-1)

for imageIdx in range(len(filenames)):
  print(filenames[imageIdx])
  for i in range(len(classes)):
      print(classes[i] + " " + str((text_probs[imageIdx][i] * 100).item()) + " %")
  print()


baddy2.png
collie 100.0 %

benji1.png
collie 100.0 %

benji_gif_2.png
collie 100.0 %

emma_all3.png
collie 100.0 %



In [22]:
for imgIdx in range(len(filenames)):
  print(filenames[imgIdx])
  for txtIdx in range(len(classes)):
    print(classes[txtIdx], cosDist(imageFeatures[imgIdx], textFeatures[txtIdx]).item())
  print()

baddy2.png
collie 0.8349609375

benji1.png
collie 0.822265625

benji_gif_2.png
collie 0.80029296875

emma_all3.png
collie 0.814453125



In [26]:
import torch.nn.functional as F

text_probs_norm = 1 - (F.normalize(textFeatures) @ F.normalize(imageFeatures).T)

print(text_probs_norm)

idx = torch.argsort(text_probs_norm)

sorted_a=torch.stack([text_probs_norm[i,idx[i]] for i in range(text_probs_norm.shape[0])] )
print(sorted_a)

print(idx)

for imageIdx in range(len(filenames)):
  print(filenames[imageIdx])
  for i in range(len(classes)):
      print(classes[i] + " " + str((text_probs_norm[i][imageIdx]).item()) + " %")
  print()


tensor([[0.8350, 0.8223, 0.8008, 0.8145]], device='cuda:0',
       dtype=torch.float16)
tensor([[0.8008, 0.8145, 0.8223, 0.8350]], device='cuda:0',
       dtype=torch.float16)
tensor([[2, 3, 1, 0]], device='cuda:0')
baddy2.png
collie 0.8349609375 %

benji1.png
collie 0.822265625 %

benji_gif_2.png
collie 0.80078125 %

emma_all3.png
collie 0.814453125 %



In [53]:
a = torch.tensor([[ 1, 1, 1]] , dtype=torch.float)
b = torch.tensor([[ 4, 7, 0]] , dtype=torch.float)
c = torch.tensor([[ 1, 2, 3]] , dtype=torch.float)
d = torch.tensor([[ -1, 1, 4]] , dtype=torch.float)
cd = torch.tensor([[ 1, 2, 3], [-1, 1, 4], [8, 8, 9], [7, 7, 4]] , dtype=torch.float)
cd_expanded = torch.unsqueeze(cd, 0)
cd_expanded2 = cd_expanded.expand(2, -1, -1)

subtractor = torch.stack([a, b])

print(cd.size())
print(cd_expanded.size())
print(cd_expanded2.size())
print(subtractor.size())
print(cd_expanded)
print(cd_expanded2)
print(subtractor)

print(torch.norm(cd_expanded2 - subtractor, dim=2).sum(1))

print(torch.norm(cd - a, dim=1).sum())
print(torch.norm(cd - b, dim=1).sum())

torch.Size([1, 4, 3])
torch.Size([2, 4, 3])
torch.Size([2, 1, 3])
tensor([[[ 1.,  2.,  3.],
         [-1.,  1.,  4.],
         [ 8.,  8.,  9.],
         [ 7.,  7.,  4.]]])
tensor([[[ 1.,  2.,  3.],
         [-1.,  1.,  4.],
         [ 8.,  8.,  9.],
         [ 7.,  7.,  4.]],

        [[ 1.,  2.,  3.],
         [-1.,  1.,  4.],
         [ 8.,  8.,  9.],
         [ 7.,  7.,  4.]]])
tensor([[[1., 1., 1.]],

        [[4., 7., 0.]]])
tensor([27.5695, 30.2319])
tensor(27.5695)
tensor(30.2319)
